In [ ]:
import os

import kagglehub

from hated_hater.paths import DATASET_DIR, RAW_DATASET_DIR

os.makedirs(RAW_DATASET_DIR, exist_ok=True)
os.makedirs(DATASET_DIR, exist_ok=True)

if not any(RAW_DATASET_DIR.iterdir()):
    kagglehub.dataset_download(
        "julian3833/jigsaw-toxic-comment-classification-challenge",
        output_dir=RAW_DATASET_DIR,
    )

In [15]:
import numpy as np
import pandas as pd

train_data: pd.DataFrame = pd.read_csv(RAW_DATASET_DIR / "train.csv", index_col="id")
test_data_comments: pd.DataFrame = pd.read_csv(RAW_DATASET_DIR / "test.csv", index_col="id")
test_labels: pd.DataFrame = pd.read_csv(RAW_DATASET_DIR / "test_labels.csv", index_col="id")

In [16]:
train_data.info()

<class 'pandas.DataFrame'>
Index: 159571 entries, 0000997932d777bf to fff46fc426af1f9a
Data columns (total 7 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   comment_text   159571 non-null  str  
 1   toxic          159571 non-null  int64
 2   severe_toxic   159571 non-null  int64
 3   obscene        159571 non-null  int64
 4   threat         159571 non-null  int64
 5   insult         159571 non-null  int64
 6   identity_hate  159571 non-null  int64
dtypes: int64(6), str(1)
memory usage: 9.7+ MB


In [17]:
test_data_comments.info()

<class 'pandas.DataFrame'>
Index: 153164 entries, 00001cee341fdb12 to ffffce3fb183ee80
Data columns (total 1 columns):
 #   Column        Non-Null Count   Dtype
---  ------        --------------   -----
 0   comment_text  153164 non-null  str  
dtypes: str(1)
memory usage: 2.3+ MB


In [18]:
test_labels.info()

<class 'pandas.DataFrame'>
Index: 153164 entries, 00001cee341fdb12 to ffffce3fb183ee80
Data columns (total 6 columns):
 #   Column         Non-Null Count   Dtype
---  ------         --------------   -----
 0   toxic          153164 non-null  int64
 1   severe_toxic   153164 non-null  int64
 2   obscene        153164 non-null  int64
 3   threat         153164 non-null  int64
 4   insult         153164 non-null  int64
 5   identity_hate  153164 non-null  int64
dtypes: int64(6)
memory usage: 8.2+ MB


In [19]:
test_data: pd.DataFrame = pd.merge(
    test_data_comments,
    test_labels,
    left_index=True,
    right_index=True,
    how="outer",
)

print(f"test_data_comments: {test_data_comments.shape}")
print(f"test_labels: {test_labels.shape}")
print(f"test_data: {test_data.shape}")

test_data_comments: (153164, 1)
test_labels: (153164, 6)
test_data: (153164, 7)


In [20]:
label_columns = [
    "toxic",
    "severe_toxic",
    "obscene",
    "threat",
    "insult",
    "identity_hate",
]
test_data = test_data[(test_data[label_columns] != -1).all(axis=1)]
print(test_data.shape)

(63978, 7)


In [21]:
test_data.head(10)

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
id,,,,,,,
0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0
000247e83dcc1211,:Dear god this site is horrible.,0,0,0,0,0,0
0002f87b16116a7f,"""::: Somebody will invariably try to add Relig...",0,0,0,0,0,0
0003e1cccfd5a40a,""" \n\n It says it right there that it IS a typ...",0,0,0,0,0,0
00059ace3e3e9a53,""" \n\n == Before adding a new product to the l...",0,0,0,0,0,0
000663aff0fffc80,this other one from 1897,0,0,0,0,0,0
000689dd34e20979,== Reason for banning throwing == \n\n This ar...,0,0,0,0,0,0
000844b52dee5f3f,|blocked]] from editing Wikipedia. |,0,0,0,0,0,0
00091c35fa9d0465,"== Arabs are committing genocide in Iraq, but ...",1,0,0,0,0,0


In [ ]:
def try_drop_id(df: pd.DataFrame) -> None:
    df.reset_index(drop=True, inplace=True)
    if "id" in df.columns:
        df.drop(columns=["id"], inplace=True)


try_drop_id(train_data)
try_drop_id(test_data)

train_data.head()

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [23]:
def int_columns_to_bool(df: pd.DataFrame, columns: list[str]) -> None:
    for column in columns:
        assert np.unique(df[column]).tolist() == [0, 1]
        df[column] = df[column].astype(bool)


int_columns_to_bool(train_data, label_columns)
int_columns_to_bool(test_data, label_columns)

train_data.head()

,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,Explanation\nWhy the edits made under my usern...,False,False,False,False,False,False
1,D'aww! He matches this background colour I'm s...,False,False,False,False,False,False
2,"Hey man, I'm really not trying to edit war. It...",False,False,False,False,False,False
3,"""\nMore\nI can't make any real suggestions on ...",False,False,False,False,False,False
4,"You, sir, are my hero. Any chance you remember...",False,False,False,False,False,False


In [24]:
train_data.to_csv(DATASET_DIR / "train.csv", index=False)
test_data.to_csv(DATASET_DIR / "test.csv", index=False)